# Notebook 2: Hierarchical Features

## Theoretical Foundation
As defined in **Section 2.10.1 (Cross-sectional hierarchical forecasting)**, we must recognize that M5 sales are naturally grouped. Forecasting at multiple levels and reconciling them improves accuracy globally.

This notebook builds the hierarchical dataset:
- **Level 0 (Total):** The sum of all sales.
- **Level 1 (Department):** Sales grouped by department (`dept_id`).
- **Level 2 (Store-Department):** Sales grouped by store and department (`store_id`, `dept_id`).
- **Level 3 (Item-Store):** The base level.

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

df = pd.read_parquet('data/01_preprocessed_m5.parquet')
print(f"Loaded {df.shape[0]} rows of preprocessed data.")

### 1. Constructing the Hierarchy

In [ ]:
# The raw data has item_id, dept_id, cat_id, store_id
print("Columns available:", df.columns.tolist())

# Level 2: Series
level_2 = df[['month', 'series_id', 'category', 'demand_clean']].copy()

# Level 1: Category (AGGREGATE ON RAW SCALE)
level_1 = level_2.groupby(['month', 'category'])['demand_clean'].sum().reset_index()
level_1['series_id'] = 'CAT_' + level_1['category']

# Level 0: Total (AGGREGATE ON RAW SCALE)
level_0 = level_2.groupby(['month'])['demand_clean'].sum().reset_index()
level_0['series_id'] = 'TOTAL'
level_0['category'] = 'TOTAL'

# Combine all levels
hierarchical_df = pd.concat([
    level_0[['month', 'series_id', 'demand_clean']],
    level_1[['month', 'series_id', 'demand_clean']],
    level_2[['month', 'series_id', 'demand_clean']]
], ignore_index=True)

# APPLY STABILIZATION (log1p) AFTER AGGREGATION
hierarchical_df['demand_transformed'] = np.log1p(hierarchical_df['demand_clean'])

print(f"Hierarchical Dataset: {hierarchical_df.shape[0]} rows across {hierarchical_df['series_id'].nunique()} distinct series.")


### 2. Feature Engineering across all levels
We engineer lags and rolling features for *every* node in the hierarchy, allowing models to learn dynamics at the aggregate and base levels.

In [ ]:
def engineer_features(group):
    g = group.sort_values('month').copy()
    target = g['demand_transformed']
    
    # Lags
    for lag in [1, 2, 3, 6, 12]:
        g[f'lag_{lag}'] = target.shift(lag)
        
    # Rolling Means
    for w in [3, 6, 12]:
        g[f'roll_mean_{w}'] = target.shift(1).rolling(w, min_periods=1).mean()
        g[f'roll_std_{w}'] = target.shift(1).rolling(w, min_periods=2).std().fillna(0)
        
    # Calendar Features
    g['month_num'] = g['month'].dt.month
    g['quarter'] = g['month'].dt.quarter
    g['month_sin'] = np.sin(2 * np.pi * g['month_num'] / 12.0)
    g['month_cos'] = np.cos(2 * np.pi * g['month_num'] / 12.0)
    
    return g

# Apply feature engineering to each series in the hierarchy
features_df = hierarchical_df.groupby('series_id').apply(engineer_features).reset_index(drop=True)

# Drop initial rows with NaNs due to max lag (12)
features_df = features_df.dropna(subset=['lag_12']).reset_index(drop=True)
print(f"Final Features Dataset: {features_df.shape[0]} rows, {features_df.shape[1]} columns.")

features_df.to_parquet('data/02_hierarchical_features.parquet', index=False)
print("Saved hierarchical features.")